특정 윈도우의 화면을 주기적으로 캡처하여 이미지 파일로 저장

In [1]:
import numpy as np  # numpy 라이브러리
import win32gui, win32ui, win32con  # 윈도우 GUI를 다루기 위한 라이브러리
from PIL import Image  # 이미지 처리를 위한 PIL 라이브러리
from time import sleep  # 시간 지연을 위한 sleep 함수
import os  # 운영체제와 상호작용하기 위한 라이브러리

In [2]:
class WindowCapture:
    w = 0  # 캡처할 윈도우의 너비
    h = 0  # 캡처할 윈도우의 높이
    hwnd = None  # 윈도우 핸들

    def __init__(self, window_name):
        self.hwnd = win32gui.FindWindow(None, window_name)  # 지정된 이름의 윈도우 핸들을 찾기
        if not self.hwnd:
            raise Exception('Window not found: {}'.format(window_name))  # 윈도우를 찾지 못하면 예외 발생

        window_rect = win32gui.GetWindowRect(self.hwnd)  # 윈도우의 좌표 가져오기
        self.w = window_rect[2] - window_rect[0]  # 윈도우의 너비 계산
        self.h = window_rect[3] - window_rect[1]  # 윈도우의 높이 계산

        # 윈도우 테두리와 제목 표시줄의 픽셀을 제외한 크기 계산
        border_pixels = 8
        titlebar_pixels = 30
        self.w = self.w - (border_pixels * 2)
        self.h = self.h - titlebar_pixels - border_pixels
        self.cropped_x = border_pixels
        self.cropped_y = titlebar_pixels

    def get_screenshot(self):
        # 윈도우의 디바이스 컨텍스트를 얻어 이미지를 캡처
        wDC = win32gui.GetWindowDC(self.hwnd)
        dcObj = win32ui.CreateDCFromHandle(wDC)
        cDC = dcObj.CreateCompatibleDC()
        dataBitMap = win32ui.CreateBitmap()
        dataBitMap.CreateCompatibleBitmap(dcObj, self.w, self.h)
        cDC.SelectObject(dataBitMap)
        cDC.BitBlt((0, 0), (self.w, self.h), dcObj, (self.cropped_x, self.cropped_y), win32con.SRCCOPY)

        signedIntsArray = dataBitMap.GetBitmapBits(True)
        img = np.fromstring(signedIntsArray, dtype='uint8')
        img.shape = (self.h, self.w, 4)

        # 사용한 리소스를 정리
        dcObj.DeleteDC()
        cDC.DeleteDC()
        win32gui.ReleaseDC(self.hwnd, wDC)
        win32gui.DeleteObject(dataBitMap.GetHandle())

        img = img[...,:3]  # 알파 채널을 제거
        img = np.ascontiguousarray(img)  # 메모리에 연속 배열로 저장
            
        return img  # 이미지 반환

    def generate_image_dataset(self):
        # 이미지 폴더가 없으면 생성
        if not os.path.exists("images"):
            os.mkdir("images")
        while(True):
            img = self.get_screenshot()  # 스크린샷을 캡처
            im = Image.fromarray(img[..., [2, 1, 0]])  # RGB 순서로 변환
            im.save(f"./images/img_{len(os.listdir('images'))}.jpg")  # 이미지 파일로 저장
            sleep(0.3)  # 0.3초 대기

    def get_window_size(self):
        return (self.w, self.h)  # 윈도우의 크기를 반환합니다.

In [7]:
# Execute this cell to generate a dataset of images for the specified window.

window_name = "BlueStacks App Player"

wincap = WindowCapture(window_name)
wincap.generate_image_dataset()

C:\Users\user\AppData\Local\Temp\ipykernel_1712\2558226797.py:32: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  img = np.fromstring(signedIntsArray, dtype='uint8')


KeyboardInterrupt: 